In [ ]:
import random
import time

class AgenteAsistencial:
    """Representa a un profesional en el modelo (Nivel Micro / Agente)"""
    def __init__(self, id_agente, rol, umbral):
        self.id = id_agente
        self.rol = rol
        self.umbral_saturacion = umbral # Límite de carga antes de pedir ayuda[cite: 4, 8]
        self.carga_actual = 0
        self.estado = "Normal" # Estados: Normal, Saturado, Ruderal (Enjambre)
        self.nivel_burnout = 0  # Desgaste acumulado por operar sobre el umbral[cite: 1, 5]

    def procesar_casos(self):
        """Simula la atención de pacientes y la retracción eficiente"""
        if self.carga_actual > 0:
            self.carga_actual -= 1 # Resuelve 1 caso por ciclo/hora

        # Tasa de retracción eficiente: Vuelve al estado base si la crisis pasa
        if self.carga_actual < self.umbral_saturacion:
            if self.estado == "Saturado":
                self.estado = "Normal"
            elif self.estado == "Ruderal (Apoyo)":
                self.estado = "Normal"

    def recibir_caso(self):
        """Asume un caso e identifica si supera su umbral"""
        self.carga_actual += 1
        if self.carga_actual >= self.umbral_saturacion:
            self.estado = "Saturado"
            self.nivel_burnout += 1 # La falta de retracción causa Burnout[cite: 5]
            return True # Retorna True para emitir "Alerta Local"
        return False

class ColmenaBienestar:
    """Representa el entorno sociotécnico de Bienestar (Nivel Macro)"""
    def __init__(self, num_medicos=2, num_psicologos=3):
        self.agentes = []
        # Parametrización de umbrales según documentos: Médicos = 4, Psicólogos = 3[cite: 4]
        for i in range(num_medicos):
            self.agentes.append(AgenteAsistencial(f"MED-{i+1}", "Médico", 4))
        for i in range(num_psicologos):
            self.agentes.append(AgenteAsistencial(f"PSI-{i+1}", "Psicólogo", 3))

        self.pacientes_en_espera = 0
        self.tiempo_objetivo_recuperacion_rto = 0 # Métrica de resiliencia[cite: 8]

    def emitir_alerta_y_enjambre(self, agente_saturado):
        """Comportamiento de Enjambre: Activación oportunista ante una alerta[cite: 5, 8]"""
        print(f"  [!] ALERTA LOCAL: {agente_saturado.id} superó su umbral ({agente_saturado.carga_actual}/{agente_saturado.umbral_saturacion}). Usando canales informales (WhatsApp)...")

        # Buscar agentes disponibles para redistribución descentralizada[cite: 5, 9]
        for compañero in self.agentes:
            if compañero.id != agente_saturado.id and compañero.estado == "Normal" and compañero.carga_actual < compañero.umbral_saturacion - 1:
                # El compañero asume el excedente cambiando a estado ruderal[cite: 5]
                compañero.estado = "Ruderal (Apoyo)"
                compañero.carga_actual += 1
                agente_saturado.carga_actual -= 1
                print(f"ENJAMBRE ACTIVADO: {compañero.id} asume carga solidaria de {agente_saturado.id}.")
                return True

        print(f"FALLO SISTÉMICO: Nadie disponible. RPO/RTO degradados y aumenta el Burnout.")
        self.pacientes_en_espera += 1
        return False

    def simular_ciclo(self, hora, casos_programados, eventos_estocasticos):
        """Ejecuta un ciclo de tiempo en la simulación (ej. 1 hora)[cite: 9, 10]"""
        print(f"\n--- HORA {hora}:00 ---")
        print(f"Demanda Programada: {casos_programados} | Eventos Estocásticos (Crisis): {eventos_estocasticos}")

        casos_totales = casos_programados + eventos_estocasticos + self.pacientes_en_espera
        self.pacientes_en_espera = 0

        # 1. Asignación inicial de casos
        for _ in range(casos_totales):
            agente_aleatorio = random.choice(self.agentes)
            alerta = agente_aleatorio.recibir_caso()

            # 2. Irrupción de perturbaciones (Ruptura de agenda)
            if alerta:
                self.emitir_alerta_y_enjambre(agente_aleatorio)

        # 3. Procesamiento y Retracción
        for agente in self.agentes:
            agente.procesar_casos()

        # 4. Reporte de Estado Macro (PySD equivalente)[cite: 10]
        print("\nESTADO DE LA COLMENA (Nivel Micro):")
        burnout_total = 0
        for ag in self.agentes:
            print(f"  {ag.id} ({ag.rol}) -> Carga: {ag.carga_actual} | Estado: {ag.estado} | Burnout: {ag.nivel_burnout}")
            burnout_total += ag.nivel_burnout

        if self.pacientes_en_espera > 0:
            self.tiempo_objetivo_recuperacion_rto += 1

        print(f"MÉTRICAS MACRO: Pacientes en Espera: {self.pacientes_en_espera} | RTO Acumulado: {self.tiempo_objetivo_recuperacion_rto} | Burnout Total: {burnout_total}")

# --- EJECUCIÓN DE LA SIMULACIÓN ---
simulador = ColmenaBienestar(num_medicos=2, num_psicologos=3)

# Escenarios de prueba (Demanda programada, Eventos estocásticos imprevistos)[cite: 1, 9]
escenarios_por_hora = [
    (3, 0), # Hora 1: Flujo regular (Agenda normal)
    (4, 1), # Hora 2: Pequeña perturbación
    (3, 8), # Hora 3: CRISIS SEVERA (Pico de ansiedad / Ruptura estocástica)
    (2, 0), # Hora 4: Estabilización (Retracción)
]

for hora, (prog, estocastico) in enumerate(escenarios_por_hora, start=8):
    simulador.simular_ciclo(hora, prog, estocastico)
    time.sleep(1.5) # Pausa para ver la salida en consola


--- HORA 8:00 ---
Demanda Programada: 3 | Eventos Estocásticos (Crisis): 0

ESTADO DE LA COLMENA (Nivel Micro):
  MED-1 (Médico) -> Carga: 0 | Estado: Normal | Burnout: 0
  MED-2 (Médico) -> Carga: 0 | Estado: Normal | Burnout: 0
  PSI-1 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
  PSI-2 (Psicólogo) -> Carga: 1 | Estado: Normal | Burnout: 0
  PSI-3 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
MÉTRICAS MACRO: Pacientes en Espera: 0 | RTO Acumulado: 0 | Burnout Total: 0

--- HORA 9:00 ---
Demanda Programada: 4 | Eventos Estocásticos (Crisis): 1
  [!] ALERTA LOCAL: PSI-1 superó su umbral (3/3). Usando canales informales (WhatsApp)...
ENJAMBRE ACTIVADO: MED-1 asume carga solidaria de PSI-1.

ESTADO DE LA COLMENA (Nivel Micro):
  MED-1 (Médico) -> Carga: 0 | Estado: Normal | Burnout: 0
  MED-2 (Médico) -> Carga: 1 | Estado: Normal | Burnout: 0
  PSI-1 (Psicólogo) -> Carga: 1 | Estado: Normal | Burnout: 1
  PSI-2 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
  PS

**Deep Q-Learning**

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta

def generar_datos_ruda(num_registros=2000):
    datos = []

    # Parámetros del modelo BPSC
    tipos_atencion = ["Programada", "Estocástica/Emergencia", "Taller P&P"]
    roles = ["Medicina", "Psicología", "Enfermería", "Trabajo Social", "Nutrición"]
    motivos = ["Salud Mental", "Lesión Física", "Trámite", "Prevención"]
    resoluciones = ["Atendido", "Remitido EPS", "Alta"]

    # Fecha de inicio de la simulación (Semestre 2026-1)[cite: 6]
    fecha_actual = datetime(2026, 2, 2, 8, 0)

    for i in range(1, num_registros + 1):
        # 1. Determinar el tipo de atención (70% programada, 25% estocástica, 5% talleres)
        tipo = random.choices(tipos_atencion, weights=[0.70, 0.25, 0.05])[0]

        # 2. Asignar rol y motivo con lógica clínica
        if tipo == "Taller P&P":
            rol = random.choice(["Psicología", "Trabajo Social", "Enfermería"])
            motivo = "Prevención"
            duracion = random.randint(60, 120)
            resolucion = "Atendido"
        elif tipo == "Estocástica/Emergencia":
            motivo = random.choices(["Salud Mental", "Lesión Física"], weights=[0.6, 0.4])[0]
            rol = "Psicología" if motivo == "Salud Mental" else "Medicina"
            duracion = random.randint(30, 90) # Las emergencias rompen la agenda por su duración[cite: 6]
            resolucion = random.choices(["Atendido", "Remitido EPS"], weights=[0.8, 0.2])[0]
        else:
            motivo = random.choices(["Trámite", "Prevención", "Salud Mental", "Lesión Física"], weights=[0.4, 0.3, 0.2, 0.1])[0]
            rol = random.choice(roles)
            duracion = random.randint(15, 45) # Tasa nominal base[cite: 6]
            resolucion = random.choices(["Atendido", "Alta"], weights=[0.7, 0.3])[0]

        # 3. Simular picos de demanda (ej. periodos de exámenes generan más llegadas simultáneas)
        salto_minutos = random.randint(0, 15) if tipo == "Estocástica/Emergencia" else random.randint(15, 30)
        fecha_actual += timedelta(minutes=salto_minutos)

        # Respetar horario laboral (8 AM a 5 PM)
        if fecha_actual.hour >= 17:
            fecha_actual += timedelta(hours=15) # Saltar al día siguiente

        datos.append({
            "ID_Registro": 1000 + i,
            "Tipo_Atencion": tipo,
            "Rol_Atiende": rol,
            "Fecha_Hora_Llegada": fecha_actual.strftime("%d/%m/%Y %H:%M"),
            "Duracion_Servicio": duracion,
            "Motivo_Consulta": motivo,
            "Resolucion": resolucion
        })

    df = pd.DataFrame(datos)
    return df

# Generar y guardar el dataset
dataset_entrenamiento = generar_datos_ruda(2000)
dataset_entrenamiento.to_csv("datos_entrenamiento_ruda.csv", index=False)
print("Dataset generado exitosamente con 2000 registros.")
print(dataset_entrenamiento.head(10))

Dataset generado exitosamente con 2000 registros.
   ID_Registro           Tipo_Atencion     Rol_Atiende Fecha_Hora_Llegada  \
0         1001              Programada      Psicología   02/02/2026 08:23   
1         1002              Taller P&P  Trabajo Social   02/02/2026 08:41   
2         1003  Estocástica/Emergencia      Psicología   02/02/2026 08:51   
3         1004              Programada      Enfermería   02/02/2026 09:17   
4         1005              Programada        Medicina   02/02/2026 09:46   
5         1006              Programada      Enfermería   02/02/2026 10:13   
6         1007              Programada      Psicología   02/02/2026 10:39   
7         1008  Estocástica/Emergencia      Psicología   02/02/2026 10:49   
8         1009              Programada        Medicina   02/02/2026 11:12   
9         1010              Programada      Enfermería   02/02/2026 11:32   

   Duracion_Servicio Motivo_Consulta    Resolucion  
0                 38         Trámite          Alt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

# 1. ARQUITECTURA DE LA RED NEURONAL (El "Cerebro" del Agente)
class RedNeuronalRUDA(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(RedNeuronalRUDA, self).__init__()
        # Capas densas para procesar las variables del modelo BPSC
        self.fc1 = nn.Linear(input_dim, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, output_dim) # Salida: Valores Q para cada acción posible

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# 2. EL ENTORNO DE SIMULACIÓN (Proceso de Decisión de Markov)
class EntornoBienestar:
    def __init__(self, umbral_saturacion=4):
        self.umbral_saturacion = umbral_saturacion
        self.reset()

    def reset(self):
        self.carga_actual = 0
        self.pacientes_espera = 0
        self.nivel_burnout = 0
        # Estado [Carga, Espera, Burnout]
        return np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)

    def step(self, accion):
        """
        Acciones (A):
        0: Atender de forma normal
        1: Activar Enjambre (Pedir ayuda) -> Lógica Ruderal (Exploración)
        2: Retracción Eficiente (Descanso) -> Lógica Ruderal (Explotación)
        """
        recompensa = 0

        # Perturbación estocástica (Adversarial RL)
        if random.random() < 0.3: # 30% de probabilidad de una crisis emergente
            self.carga_actual += random.randint(1, 3)
            self.pacientes_espera += random.randint(0, 2)

        # Resolución de la acción
        if accion == 0: # Atender Normal
            if self.carga_actual > 0:
                self.carga_actual -= 1
                recompensa += 10 # Recompensa por resolver casos
            if self.carga_actual >= self.umbral_saturacion:
                self.nivel_burnout += 1
                recompensa -= 20 # Penalización severa por saturarse sin pedir ayuda

        elif accion == 1: # Activar Enjambre
            if self.carga_actual >= self.umbral_saturacion:
                self.carga_actual = max(0, self.carga_actual - 2) # Compañeros asumen carga
                recompensa += 15 # Recompensa por delegar a tiempo (Supervivencia del sistema)
            else:
                recompensa -= 10 # Penalización por pedir ayuda sin necesitarla (Falsa alarma)

        elif accion == 2: # Retracción Eficiente
            if self.carga_actual < self.umbral_saturacion:
                self.nivel_burnout = max(0, self.nivel_burnout - 1)
                recompensa += 10 # Recompensa por recuperación (Bajar RTO/Burnout)
            else:
                self.nivel_burnout += 2
                self.pacientes_espera += 1
                recompensa -= 20 # Penalización por descansar en medio de una crisis

        # Penalizaciones globales del entorno (RTO y RPO degradados)
        recompensa -= (self.pacientes_espera * 2)
        recompensa -= (self.nivel_burnout * 2)

        nuevo_estado = np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)
        done = self.nivel_burnout >= 10 # Condición de fallo sistémico total

        return nuevo_estado, recompensa, done

# 3. EL AGENTE DE APRENDIZAJE POR REFUERZO (DQN Agent)
class AgenteRUDA:
    def __init__(self, state_dim, action_dim):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.memoria = deque(maxlen=2000)
        self.gamma = 0.95    # Factor de descuento (visión a largo plazo)
        self.epsilon = 1.0   # Tasa de exploración inicial
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001

        self.modelo = RedNeuronalRUDA(state_dim, action_dim)
        self.optimizador = optim.Adam(self.modelo.parameters(), lr=self.learning_rate)
        self.criterio = nn.MSELoss()

    def actuar(self, estado):
        # Dilema Exploración-Explotación
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_dim) # Acción estocástica aleatoria
        estado_tensor = torch.FloatTensor(estado)
        con_gradientes = self.modelo(estado_tensor)
        return torch.argmax(con_gradientes).item() # Mejor acción predicha por la Red Neuronal

    def recordar(self, estado, accion, recompensa, siguiente_estado, done):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, done))

    def entrenar_experiencia(self, batch_size):
        if len(self.memoria) < batch_size:
            return

        minibatch = random.sample(self.memoria, batch_size)

        for estado, accion, recompensa, siguiente_estado, done in minibatch:
            objetivo = recompensa
            if not done:
                siguiente_estado_tensor = torch.FloatTensor(siguiente_estado)
                objetivo = recompensa + self.gamma * torch.max(self.modelo(siguiente_estado_tensor)).item()

            estado_tensor = torch.FloatTensor(estado)
            q_values = self.modelo(estado_tensor)
            q_values_target = q_values.clone()
            q_values_target[accion] = objetivo # Actualiza el valor Q esperado

            # Retropropagación del error para ajustar los pesos de la neurona
            self.optimizador.zero_grad()
            perdida = self.criterio(q_values, q_values_target)
            perdida.backward()
            self.optimizador.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# --- EJECUCIÓN DEL ENTRENAMIENTO (Episodios) ---
if __name__ == "__main__":
    entorno = EntornoBienestar(umbral_saturacion=4)
    agente = AgenteRUDA(state_dim=3, action_dim=3)
    episodes = 500
    batch_size = 32

    print("Iniciando entrenamiento bioinspirado del Enjambre (RL + Red Neuronal)...")
    for e in range(episodes):
        estado = entorno.reset()
        recompensa_total = 0

        for tiempo in range(50): # Cada episodio simula 50 ciclos u horas de trabajo
            accion = agente.actuar(estado)
            siguiente_estado, recompensa, done = entorno.step(accion)

            agente.recordar(estado, accion, recompensa, siguiente_estado, done)
            estado = siguiente_estado
            recompensa_total += recompensa

            if done:
                print(f"Episodio: {e+1}/{episodes} | COLAPSO en t={tiempo} | Recompensa: {recompensa_total} | Epsilon: {agente.epsilon:.2f}")
                break

        agente.entrenar_experiencia(batch_size)

        if (e+1) % 50 == 0 and not done:
             print(f"Episodio: {e+1}/{episodes} | ÉXITO (Sin colapso) | Recompensa: {recompensa_total} | Epsilon: {agente.epsilon:.2f}")

Iniciando entrenamiento bioinspirado del Enjambre (RL + Red Neuronal)...
Episodio: 1/500 | COLAPSO en t=8 | Recompensa: -253 | Epsilon: 1.00
Episodio: 2/500 | COLAPSO en t=24 | Recompensa: -614 | Epsilon: 1.00
Episodio: 3/500 | COLAPSO en t=30 | Recompensa: -728 | Epsilon: 0.99
Episodio: 4/500 | COLAPSO en t=43 | Recompensa: -773 | Epsilon: 0.99
Episodio: 5/500 | COLAPSO en t=26 | Recompensa: -304 | Epsilon: 0.99
Episodio: 7/500 | COLAPSO en t=22 | Recompensa: -499 | Epsilon: 0.98
Episodio: 9/500 | COLAPSO en t=43 | Recompensa: -1279 | Epsilon: 0.97
Episodio: 10/500 | COLAPSO en t=28 | Recompensa: -520 | Epsilon: 0.96
Episodio: 12/500 | COLAPSO en t=24 | Recompensa: -533 | Epsilon: 0.95
Episodio: 15/500 | COLAPSO en t=32 | Recompensa: -482 | Epsilon: 0.94
Episodio: 17/500 | COLAPSO en t=28 | Recompensa: -448 | Epsilon: 0.93
Episodio: 19/500 | COLAPSO en t=44 | Recompensa: -1267 | Epsilon: 0.92
Episodio: 20/500 | COLAPSO en t=8 | Recompensa: -302 | Epsilon: 0.91
Episodio: 21/500 | COLAP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
from collections import deque
import os

# 1. ARQUITECTURA DE LA RED NEURONAL (Mantiene la estructura original)
class RedNeuronalRUDA(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(RedNeuronalRUDA, self).__init__()
        self.fc1 = nn.Linear(input_dim, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# 2. ENTORNO ALIMENTADO POR DATOS (Sustituye la probabilidad aleatoria por el Dataset)
class EntornoDatosBienestar:
    def __init__(self, ruta_dataset, umbral_saturacion=4):
        self.umbral_saturacion = umbral_saturacion
        # Cargar el dataset sintético de la Unidad de Bienestar[cite: 6]
        if not os.path.exists(ruta_dataset):
            raise FileNotFoundError(f"No se encontró el archivo: {ruta_dataset}. Ejecuta primero el generador de datos.")
        self.dataset = pd.read_csv(ruta_dataset)
        self.total_registros = len(self.dataset)
        self.reset()

    def reset(self):
        self.carga_actual = 0
        self.pacientes_espera = 0
        self.nivel_burnout = 0
        self.indice_datos = 0 # Puntero para recorrer el dataset fila por fila
        return np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)

    def step(self, accion):
        recompensa = 0

        # --- LECTURA DEL DATASET (Inyección de la perturbación real) ---
        if self.indice_datos < self.total_registros:
            fila_actual = self.dataset.iloc[self.indice_datos]
            tipo_atencion = fila_actual['Tipo_Atencion']
            duracion = fila_actual['Duracion_Servicio']

            # Codificación del impacto según el tipo de atención[cite: 6]
            if tipo_atencion == "Estocástica/Emergencia":
                self.carga_actual += 2 # La emergencia pesa el doble y rompe la agenda
                self.pacientes_espera += 1
            elif tipo_atencion == "Programada":
                self.carga_actual += 1
            elif tipo_atencion == "Taller P&P":
                self.carga_actual += 1

            self.indice_datos += 1

        # --- RESOLUCIÓN DE LA ACCIÓN DEL AGENTE ---
        if accion == 0: # Atender Normal
            if self.carga_actual > 0:
                self.carga_actual = max(0, self.carga_actual - 1)
                recompensa += 10
            if self.carga_actual >= self.umbral_saturacion:
                self.nivel_burnout += 1
                recompensa -= 20

        elif accion == 1: # Activar Enjambre (Lógica Ruderal)
            if self.carga_actual >= self.umbral_saturacion:
                self.carga_actual = max(0, self.carga_actual - 2)
                recompensa += 15
            else:
                recompensa -= 10

        elif accion == 2: # Retracción Eficiente
            if self.carga_actual < self.umbral_saturacion:
                self.nivel_burnout = max(0, self.nivel_burnout - 1)
                recompensa += 10
            else:
                self.nivel_burnout += 2
                self.pacientes_espera += 1
                recompensa -= 20

        # Penalizaciones globales
        recompensa -= (self.pacientes_espera * 2)
        recompensa -= (self.nivel_burnout * 2)

        nuevo_estado = np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)

        # El episodio termina si hay colapso sistémico o si se acaba el dataset
        done = self.nivel_burnout >= 10 or self.indice_datos >= self.total_registros

        return nuevo_estado, recompensa, done

# 3. AGENTE DQN (Mantiene la lógica de memoria y retropropagación del error)
class AgenteRUDA:
    def __init__(self, state_dim, action_dim):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.memoria = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001

        self.modelo = RedNeuronalRUDA(state_dim, action_dim)
        self.optimizador = optim.Adam(self.modelo.parameters(), lr=self.learning_rate)
        self.criterio = nn.MSELoss()

    def actuar(self, estado):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_dim)
        estado_tensor = torch.FloatTensor(estado)
        con_gradientes = self.modelo(estado_tensor)
        return torch.argmax(con_gradientes).item()

    def recordar(self, estado, accion, recompensa, siguiente_estado, done):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, done))

    def entrenar_experiencia(self, batch_size):
        if len(self.memoria) < batch_size:
            return

        minibatch = random.sample(self.memoria, batch_size)

        for estado, accion, recompensa, siguiente_estado, done in minibatch:
            objetivo = recompensa
            if not done:
                siguiente_estado_tensor = torch.FloatTensor(siguiente_estado)
                objetivo = recompensa + self.gamma * torch.max(self.modelo(siguiente_estado_tensor)).item()

            estado_tensor = torch.FloatTensor(estado)
            q_values = self.modelo(estado_tensor)
            q_values_target = q_values.clone()
            q_values_target[accion] = objetivo

            self.optimizador.zero_grad()
            perdida = self.criterio(q_values, q_values_target)
            perdida.backward()
            self.optimizador.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# --- EJECUCIÓN DEL ENTRENAMIENTO ---
if __name__ == "__main__":
    # Asegúrate de que el CSV esté en la misma carpeta donde ejecutas el script
    archivo_csv = "datos_entrenamiento_ruda.csv"

    try:
        entorno = EntornoDatosBienestar(ruta_dataset=archivo_csv, umbral_saturacion=4)
        agente = AgenteRUDA(state_dim=3, action_dim=3)
        episodes = 50
        batch_size = 32

        print(f"Iniciando entrenamiento con dataset: {archivo_csv}...")
        for e in range(episodes):
            estado = entorno.reset()
            recompensa_total = 0

            # Simulamos hasta que el entorno emita el flag 'done' (Colapso o fin de datos)
            while True:
                accion = agente.actuar(estado)
                siguiente_estado, recompensa, done = entorno.step(accion)

                agente.recordar(estado, accion, recompensa, siguiente_estado, done)
                estado = siguiente_estado
                recompensa_total += recompensa

                if done:
                    motivo_fin = "FIN DEL DATASET" if entorno.indice_datos >= entorno.total_registros else "COLAPSO POR BURNOUT"
                    print(f"Episodio: {e+1}/{episodes} | Terminado por: {motivo_fin} en t={entorno.indice_datos} | Recompensa: {recompensa_total} | Epsilon: {agente.epsilon:.2f}")
                    break

            agente.entrenar_experiencia(batch_size)

    except FileNotFoundError as e:
        print(e)

Iniciando entrenamiento con dataset: datos_entrenamiento_ruda.csv...
Episodio: 1/50 | Terminado por: COLAPSO POR BURNOUT en t=18 | Recompensa: -171 | Epsilon: 1.00
Episodio: 2/50 | Terminado por: COLAPSO POR BURNOUT en t=41 | Recompensa: -698 | Epsilon: 1.00
Episodio: 3/50 | Terminado por: COLAPSO POR BURNOUT en t=18 | Recompensa: -268 | Epsilon: 0.99
Episodio: 4/50 | Terminado por: COLAPSO POR BURNOUT en t=26 | Recompensa: -373 | Epsilon: 0.99
Episodio: 5/50 | Terminado por: COLAPSO POR BURNOUT en t=18 | Recompensa: -201 | Epsilon: 0.99
Episodio: 6/50 | Terminado por: COLAPSO POR BURNOUT en t=8 | Recompensa: -236 | Epsilon: 0.98
Episodio: 7/50 | Terminado por: COLAPSO POR BURNOUT en t=11 | Recompensa: -180 | Epsilon: 0.98
Episodio: 8/50 | Terminado por: COLAPSO POR BURNOUT en t=9 | Recompensa: -233 | Epsilon: 0.97
Episodio: 9/50 | Terminado por: COLAPSO POR BURNOUT en t=20 | Recompensa: -235 | Epsilon: 0.97
Episodio: 10/50 | Terminado por: COLAPSO POR BURNOUT en t=14 | Recompensa: -18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import os
from collections import deque

# 1. ARQUITECTURA DE LA RED NEURONAL (El Motor Cognitivo RUDA)[cite: 5, 8]
class RedNeuronalRUDA(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(RedNeuronalRUDA, self).__init__()
        # Procesamiento del contexto BPSC: [Carga, Espera, Burnout][cite: 8]
        self.fc1 = nn.Linear(input_dim, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# 2. ENTORNO BPSC BASADO EN DATOS (Sustituye la probabilidad aleatoria)[cite: 4, 8]
class EntornoDatosBienestar:
    def __init__(self, ruta_dataset, umbral_saturacion=4):
        self.umbral_saturacion = umbral_saturacion

        # Validar y cargar el dataset 2026-1 de Bienestar[cite: 4, 8]
        if not os.path.exists(ruta_dataset):
            raise FileNotFoundError(f"[ERROR] No se encontró la línea base operativa: {ruta_dataset}")

        self.dataset = pd.read_csv(ruta_dataset)
        self.total_registros = len(self.dataset)
        self.reset()

    def reset(self):
        self.carga_actual = 0
        self.pacientes_espera = 0
        self.nivel_burnout = 0
        self.indice_datos = 0 # Puntero temporal del historial
        return np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)

    def step(self, accion):
        recompensa = 0

        # --- INYECCIÓN DE LA DEMANDA REAL (Línea Base OE2) ---[cite: 8]
        if self.indice_datos < self.total_registros:
            fila_actual = self.dataset.iloc[self.indice_datos]
            tipo_atencion = fila_actual['Tipo_Atencion']
            duracion = fila_actual['Duracion_Servicio']

            # Codificar el impacto estocástico de la perturbación[cite: 8]
            if tipo_atencion == "Estocástica/Emergencia":
                self.carga_actual += 2 # Las urgencias superan la capacidad estática[cite: 10]
                self.pacientes_espera += 1
            else: # Demanda programada regular[cite: 8]
                self.carga_actual += 1

            self.indice_datos += 1

        # --- RESOLUCIÓN Y EVALUACIÓN DE LA ACCIÓN DE RUDA ---[cite: 5]
        # Acciones: 0 = Operación Base, 1 = Activar Enjambre, 2 = Retracción Eficiente

        if accion == 0:
            if self.carga_actual > 0:
                self.carga_actual = max(0, self.carga_actual - 1)
                recompensa += 10 # Se atiende sin interrumpir el flujo
            if self.carga_actual >= self.umbral_saturacion:
                self.nivel_burnout += 1
                recompensa -= 20 # Castigo por omitir el comportamiento de enjambre[cite: 5, 8]

        elif accion == 1:
            if self.carga_actual >= self.umbral_saturacion:
                self.carga_actual = max(0, self.carga_actual - 2) # Redistribución oportunista[cite: 5, 8]
                recompensa += 15
            else:
                recompensa -= 10 # Falsa alarma en canales informales

        elif accion == 2:
            if self.carga_actual < self.umbral_saturacion:
                self.nivel_burnout = max(0, self.nivel_burnout - 1)
                recompensa += 10 # Retracción eficiente lograda[cite: 5, 8]
            else:
                self.nivel_burnout += 2
                self.pacientes_espera += 1
                recompensa -= 20 # Degradación del sistema si descansa durante crisis[cite: 10]

        # Métrica de penalización sistémica: Impacto negativo en RTO y RPO[cite: 8]
        recompensa -= (self.pacientes_espera * 2)
        recompensa -= (self.nivel_burnout * 2)

        nuevo_estado = np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)
        done = self.nivel_burnout >= 10 or self.indice_datos >= self.total_registros

        return nuevo_estado, recompensa, done

# 3. AGENTE MULTI-AGENTE (DQN con Ecosistema de PyTorch)[cite: 5, 8]
class AgenteRUDA:
    def __init__(self, state_dim, action_dim):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.memoria = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001

        self.modelo = RedNeuronalRUDA(state_dim, action_dim)
        self.optimizador = optim.Adam(self.modelo.parameters(), lr=self.learning_rate)
        self.criterio = nn.MSELoss() # Función de error para calibrar pesos[cite: 4]

    def actuar(self, estado):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_dim)
        estado_tensor = torch.FloatTensor(estado)
        con_gradientes = self.modelo(estado_tensor)
        return torch.argmax(con_gradientes).item()

    def recordar(self, estado, accion, recompensa, siguiente_estado, done):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, done))

    def entrenar_experiencia(self, batch_size):
        if len(self.memoria) < batch_size:
            return

        minibatch = random.sample(self.memoria, batch_size)

        for estado, accion, recompensa, siguiente_estado, done in minibatch:
            objetivo = recompensa
            if not done:
                siguiente_estado_tensor = torch.FloatTensor(siguiente_estado)
                objetivo = recompensa + self.gamma * torch.max(self.modelo(siguiente_estado_tensor)).item()

            estado_tensor = torch.FloatTensor(estado)
            q_values = self.modelo(estado_tensor)
            q_values_target = q_values.clone()
            q_values_target[accion] = objetivo

            self.optimizador.zero_grad()
            perdida = self.criterio(q_values, q_values_target)
            perdida.backward()
            self.optimizador.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# --- CICLO DE APRENDIZAJE Y VALIDACIÓN (Fase 4 - DSR) ---[cite: 5]
if __name__ == "__main__":
    archivo_csv = "datos_entrenamiento_ruda.csv"

    try:
        # Configuración del umbral operativo (ej. Médicos=4)[cite: 7]
        entorno = EntornoDatosBienestar(ruta_dataset=archivo_csv, umbral_saturacion=4)
        agente = AgenteRUDA(state_dim=3, action_dim=3)
        episodes = 100
        batch_size = 32

        print(f"--- INICIANDO APRENDIZAJE DE LA IA BIOINSPIRADA ---")
        print(f"Cargando línea base: {archivo_csv} | Registros Totales: {entorno.total_registros}")

        for e in range(episodes):
            estado = entorno.reset()
            recompensa_total = 0

            while True:
                accion = agente.actuar(estado)
                siguiente_estado, recompensa, done = entorno.step(accion)

                agente.recordar(estado, accion, recompensa, siguiente_estado, done)
                estado = siguiente_estado
                recompensa_total += recompensa

                if done:
                    motivo_fin = "FIN DE LA LÍNEA BASE (100% Analizado)" if entorno.indice_datos >= entorno.total_registros else "FALLO SISTÉMICO (Burnout > 10)"
                    print(f"Época {e+1}/{episodes} | Finaliza por: {motivo_fin} en t={entorno.indice_datos} | Eficiencia Operativa: {recompensa_total} | Tasa de Exploración: {agente.epsilon:.2f}")
                    break

            agente.entrenar_experiencia(batch_size)

    except FileNotFoundError as e:
        print(e)

--- INICIANDO APRENDIZAJE DE LA IA BIOINSPIRADA ---
Cargando línea base: datos_entrenamiento_ruda.csv | Registros Totales: 2000
Época 1/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=22 | Eficiencia Operativa: -191 | Tasa de Exploración: 1.00
Época 2/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=13 | Eficiencia Operativa: -202 | Tasa de Exploración: 1.00
Época 3/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=17 | Eficiencia Operativa: -258 | Tasa de Exploración: 0.99
Época 4/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=26 | Eficiencia Operativa: -256 | Tasa de Exploración: 0.99
Época 5/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=25 | Eficiencia Operativa: -241 | Tasa de Exploración: 0.99
Época 6/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=13 | Eficiencia Operativa: -290 | Tasa de Exploración: 0.98
Época 7/100 | Finaliza por: FALLO SISTÉMICO (Burnout > 10) en t=22 | Eficiencia Operativa: -246 | Tasa de Exploración: 0.98
Époc

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import time
from collections import deque

# =====================================================================
# PARTE 1: ARQUITECTURA DE LA IA BIOINSPIRADA (APRENDIZAJE POR REFUERZO)
# =====================================================================
class RedNeuronalRUDA(nn.Module):
    """El cerebro matemático que procesa las variables de contexto BPSC"""
    def __init__(self, input_dim, output_dim):
        super(RedNeuronalRUDA, self).__init__()
        self.fc1 = nn.Linear(input_dim, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class EntornoEntrenamiento:
    """Entorno Data-Driven para entrenar a la IA antes de la simulación"""
    def __init__(self, dataset, umbral=4):
        self.dataset = dataset
        self.total_registros = len(self.dataset)
        self.umbral_saturacion = umbral
        self.reset()

    def reset(self):
        self.carga_actual = 0
        self.pacientes_espera = 0
        self.nivel_burnout = 0
        self.indice = 0
        return np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)

    def step(self, accion):
        recompensa = 0

        # Lectura de registros del dataset
        if self.indice < self.total_registros:
            tipo = self.dataset.iloc[self.indice]['Tipo_Atencion']
            if tipo == "Estocástica/Emergencia":
                self.carga_actual += 2
                self.pacientes_espera += 1
            else:
                self.carga_actual += 1
            self.indice += 1

        # Evaluación de la Acción (0=Atender, 1=Alerta Enjambre, 2=Retracción)
        if accion == 0:
            if self.carga_actual > 0: self.carga_actual = max(0, self.carga_actual - 1)
            recompensa += 10 if self.carga_actual < self.umbral_saturacion else -20
        elif accion == 1:
            if self.carga_actual >= self.umbral_saturacion:
                self.carga_actual = max(0, self.carga_actual - 2)
                recompensa += 15
            else:
                recompensa -= 10
        elif accion == 2:
            if self.carga_actual < self.umbral_saturacion:
                self.nivel_burnout = max(0, self.nivel_burnout - 1)
                recompensa += 10
            else:
                self.nivel_burnout += 2
                self.pacientes_espera += 1
                recompensa -= 20

        # Castigos globales
        recompensa -= (self.pacientes_espera * 2 + self.nivel_burnout * 2)
        estado = np.array([self.carga_actual, self.pacientes_espera, self.nivel_burnout], dtype=np.float32)
        done = self.nivel_burnout >= 10 or self.indice >= self.total_registros
        return estado, recompensa, done

class MotorCognitivoRUDA:
    """Agente DQN que gestiona la red neuronal y la memoria de experiencia"""
    def __init__(self, state_dim, action_dim):
        self.action_dim = action_dim
        self.memoria = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.modelo = RedNeuronalRUDA(state_dim, action_dim)
        self.optimizador = optim.Adam(self.modelo.parameters(), lr=0.001)
        self.criterio = nn.MSELoss()

    def actuar(self, estado, en_simulacion=False):
        # Si está en simulación real, no explora al azar, usa el aprendizaje adquirido
        if not en_simulacion and np.random.rand() <= self.epsilon:
            return random.randrange(self.action_dim)

        estado_tensor = torch.FloatTensor(estado)
        with torch.no_grad():
            valores_q = self.modelo(estado_tensor)
        return torch.argmax(valores_q).item()

    def recordar_y_entrenar(self, estado, accion, recompensa, siguiente_estado, done, batch_size=32):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, done))
        if len(self.memoria) < batch_size: return

        minibatch = random.sample(self.memoria, batch_size)
        for e, a, r, s_e, d in minibatch:
            objetivo = r
            if not d:
                objetivo = r + self.gamma * torch.max(self.modelo(torch.FloatTensor(s_e))).item()

            q_values = self.modelo(torch.FloatTensor(e))
            q_values_target = q_values.clone()
            q_values_target[a] = objetivo

            self.optimizador.zero_grad()
            perdida = self.criterio(q_values, q_values_target)
            perdida.backward()
            self.optimizador.step()

        if self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

# =====================================================================
# PARTE 2: SIMULADOR MULTI-AGENTE (NIVEL MICRO) GOBERNADO POR LA IA
# =====================================================================
class AgenteAsistencial:
    def __init__(self, id_agente, rol, umbral, motor_ia):
        self.id = id_agente
        self.rol = rol
        self.umbral_saturacion = umbral
        self.carga_actual = 0
        self.estado = "Normal"
        self.nivel_burnout = 0
        self.motor_ia = motor_ia # La red neuronal inyectada al agente

    def recibir_caso(self, pacientes_espera_global, estocastico=False):
        self.carga_actual += 2 if estocastico else 1

        # 🧠 LA IA EVALÚA EL CONTEXTO EN LUGAR DE UN CONDICIONAL FIJO
        contexto = np.array([self.carga_actual, pacientes_espera_global, self.nivel_burnout], dtype=np.float32)
        decision = self.motor_ia.actuar(contexto, en_simulacion=True)

        if decision == 1: # La IA predice colapso y detona Alerta Local (Lógica Ruderal)
            self.estado = "Saturado"
            self.nivel_burnout += 1
            return True
        return False

    def procesar_casos(self, pacientes_espera_global):
        if self.carga_actual > 0: self.carga_actual -= 1

        # 🧠 LA IA DECIDE SI ES SEGURO EJECUTAR LA RETRACCIÓN EFICIENTE
        contexto = np.array([self.carga_actual, pacientes_espera_global, self.nivel_burnout], dtype=np.float32)
        decision = self.motor_ia.actuar(contexto, en_simulacion=True)

        if decision == 2 and self.carga_actual < self.umbral_saturacion:
            if self.estado in ["Saturado", "Ruderal (Apoyo)"]:
                self.estado = "Normal"
                self.nivel_burnout = max(0, self.nivel_burnout - 1)

class ColmenaBienestar:
    def __init__(self, motor_ia, num_medicos=2, num_psicologos=3):
        self.agentes = [AgenteAsistencial(f"MED-{i+1}", "Médico", 4, motor_ia) for i in range(num_medicos)]
        self.agentes += [AgenteAsistencial(f"PSI-{i+1}", "Psicólogo", 3, motor_ia) for i in range(num_psicologos)]
        self.pacientes_en_espera = 0
        self.fallos_sistemicos = 0

    def emitir_alerta_y_enjambre(self, agente_saturado):
        print(f"  [!] ALERTA IA: {agente_saturado.id} detecta riesgo. IA detona canales informales.")
        for compañero in self.agentes:
            if compañero.id != agente_saturado.id and compañero.estado == "Normal" and compañero.carga_actual < compañero.umbral_saturacion - 1:
                compañero.estado = "Ruderal (Apoyo)"
                compañero.carga_actual += 1
                agente_saturado.carga_actual -= 1
                print(f"  [🐝] ENJAMBRE ACTIVADO: {compañero.id} asume carga solidaria de {agente_saturado.id}.")
                return True

        print(f"  [💥] FALLO SISTÉMICO: Colmena saturada. Aumentan casos represados.")
        self.pacientes_en_espera += 1
        self.fallos_sistemicos += 1
        return False

    def simular_ciclo(self, hora, demanda_regular, eventos_estocasticos):
        print(f"\n--- HORA {hora}:00 ---")
        print(f"Llegadas -> Programada: {demanda_regular} | Estocástica (Crisis): {eventos_estocasticos}")

        for _ in range(demanda_regular):
            if random.choice(self.agentes).recibir_caso(self.pacientes_en_espera, estocastico=False):
                self.emitir_alerta_y_enjambre(random.choice(self.agentes))

        for _ in range(eventos_estocasticos):
            if random.choice(self.agentes).recibir_caso(self.pacientes_en_espera, estocastico=True):
                self.emitir_alerta_y_enjambre(random.choice(self.agentes))

        for agente in self.agentes:
            agente.procesar_casos(self.pacientes_en_espera)

        print("\nESTADO DE LA COLMENA (Decisiones ejecutadas por la IA):")
        burnout_total = 0
        for ag in self.agentes:
            print(f"  {ag.id} ({ag.rol}) -> Carga: {ag.carga_actual} | Estado: {ag.estado} | Burnout: {ag.nivel_burnout}")
            burnout_total += ag.nivel_burnout

        print(f"MÉTRICAS: Casos Represados: {self.pacientes_en_espera} | Fallos Sistémicos: {self.fallos_sistemicos} | Burnout Total: {burnout_total}")

# =====================================================================
# EJECUCIÓN MAESTRA (ENTRENAMIENTO + SIMULACIÓN)
# =====================================================================
if __name__ == "__main__":
    # 1. Generación de dataset sintético para el entrenamiento
    datos = [{"Tipo_Atencion": random.choices(["Programada", "Estocástica/Emergencia"], weights=[0.7, 0.3])[0]} for _ in range(800)]
    df_entrenamiento = pd.DataFrame(datos)

    motor_ia = MotorCognitivoRUDA(state_dim=3, action_dim=3)
    entorno_entrenamiento = EntornoEntrenamiento(df_entrenamiento)

    print("=== FASE 1: ENTRENANDO LA RED NEURONAL CON EL DATASET ===")
    for epoca in range(30):
        estado = entorno_entrenamiento.reset()
        while True:
            accion = motor_ia.actuar(estado, en_simulacion=False)
            siguiente_estado, recompensa, done = entorno_entrenamiento.step(accion)
            motor_ia.recordar_y_entrenar(estado, accion, recompensa, siguiente_estado, done)
            estado = siguiente_estado
            if done: break
    print("✓ Entrenamiento completado. La IA ha calibrado sus pesos.\n")

    print("=== FASE 2: INICIANDO SIMULACIÓN CON LA IA AL MANDO ===")
    simulador = ColmenaBienestar(motor_ia=motor_ia, num_medicos=2, num_psicologos=3)

    # Escenarios: (Carga Regular, Picos de Crisis)
    agenda_del_dia = [
        (3, 0), # 08:00 - Normalidad
        (4, 1), # 09:00 - Leve perturbación
        (2, 7), # 10:00 - CRISIS SEVERA (Sobrecarga estocástica)
        (2, 0), # 11:00 - Estabilización (La IA debe ejecutar Retracción)
    ]

    for hora, (reg, estocastico) in enumerate(agenda_del_dia, start=8):
        simulador.simular_ciclo(hora, reg, estocastico)
        time.sleep(1.5)

=== FASE 1: ENTRENANDO LA RED NEURONAL CON EL DATASET ===
✓ Entrenamiento completado. La IA ha calibrado sus pesos.

=== FASE 2: INICIANDO SIMULACIÓN CON LA IA AL MANDO ===

--- HORA 8:00 ---
Llegadas -> Programada: 3 | Estocástica (Crisis): 0

ESTADO DE LA COLMENA (Decisiones ejecutadas por la IA):
  MED-1 (Médico) -> Carga: 0 | Estado: Normal | Burnout: 0
  MED-2 (Médico) -> Carga: 0 | Estado: Normal | Burnout: 0
  PSI-1 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
  PSI-2 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
  PSI-3 (Psicólogo) -> Carga: 0 | Estado: Normal | Burnout: 0
MÉTRICAS: Casos Represados: 0 | Fallos Sistémicos: 0 | Burnout Total: 0

--- HORA 9:00 ---
Llegadas -> Programada: 4 | Estocástica (Crisis): 1
  [!] ALERTA IA: MED-2 detecta riesgo. IA detona canales informales.
  [🐝] ENJAMBRE ACTIVADO: MED-1 asume carga solidaria de MED-2.
  [!] ALERTA IA: MED-1 detecta riesgo. IA detona canales informales.
  [🐝] ENJAMBRE ACTIVADO: MED-2 asume carga solidari